# Medical LLM QLoRA cloud runner

This notebook contains no training logic. It clones the repository, reads an optional `HF_TOKEN` from notebook secrets, runs a 10-step smoke test, then runs the measured pipeline. Use a GPU runtime.

In [1]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/Leng-Bu-Ding/medical-llm-qlora.git'
base = Path('/content') if Path('/content').exists() else Path('/kaggle/working')
repo = base / 'medical-llm-qlora'
if (repo / '.git').exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
os.chdir(repo)
print(repo)

/content/medical-llm-qlora


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-train.txt'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-r', 'requirements-train.txt'], returncode=0)

In [3]:
# Optional read-only Hugging Face token. Never print or commit it.
token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if token:
    os.environ['HF_TOKEN'] = token
print('HF_TOKEN configured:', bool(token))

HF_TOKEN configured: False


In [4]:
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GiB')

Tesla T4 14.56 GiB


In [5]:
import subprocess

subprocess.run(
    ["git", "fetch", "origin", "agent/fix-unsloth-eos"],
    check=True,
)
subprocess.run(
    ["git", "checkout", "agent/fix-unsloth-eos"],
    check=True,
)

CompletedProcess(args=['git', 'checkout', 'agent/fix-unsloth-eos'], returncode=0)

In [6]:
# Mandatory 10-step validation before spending hours on the full run.
subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--run-id', 'smoke_clean', '--protocol', 'clean', '--smoke'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_pipeline.py', '--run-id', 'smoke_clean', '--protocol', 'clean', '--smoke'], returncode=0)

In [7]:
#### 报错测试

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "scripts/run_pipeline.py",
        "--run-id", "smoke_clean",
        "--protocol", "clean",
        "--smoke",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout[-30000:])
print("\nEXIT CODE:", result.returncode)

+ /usr/bin/python3 scripts/prepare_data.py --config /content/medical-llm-qlora/configs/qlora_llama3_8b.yaml --protocol clean --output /content/medical-llm-qlora/outputs/smoke_clean/data
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(
{
  "status": "measured_data_pr

In [12]:
# Change to True only after the smoke test succeeds.
RUN_FULL = True
RUN_ID = 'clean_main_v1'
if RUN_FULL:
    subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--run-id', RUN_ID, '--protocol', 'clean'], check=True)

CalledProcessError: Command '['/usr/bin/python3', 'scripts/run_pipeline.py', '--run-id', 'clean_main_v1', '--protocol', 'clean']' returned non-zero exit status 1.

In [13]:
from pathlib import Path

run_dir = Path("/content/medical-llm-qlora/outputs/clean_main_v1")

files = [
    "data/data_manifest.json",
    "training/training_summary.json",
    "predictions_300.jsonl",
    "evaluation_summary.json",
    "safety_predictions.jsonl",
    "safety_summary.json",
    "pipeline_manifest.json",
]

for name in files:
    path = run_dir / name
    if path.exists():
        print(f"✅ {name}: {path.stat().st_size:,} bytes")
    else:
        print(f"❌ {name}: missing")

checkpoint_dir = run_dir / "training" / "checkpoints"
checkpoints = (
    sorted(checkpoint_dir.glob("checkpoint-*"))
    if checkpoint_dir.exists()
    else []
)

print("\nCHECKPOINTS:")
for path in checkpoints:
    print(path.name)

✅ data/data_manifest.json: 3,326 bytes
✅ training/training_summary.json: 1,551 bytes
✅ predictions_300.jsonl: 734,285 bytes
❌ evaluation_summary.json: missing
❌ safety_predictions.jsonl: missing
❌ safety_summary.json: missing
❌ pipeline_manifest.json: missing

CHECKPOINTS:
checkpoint-1400
checkpoint-1439


In [11]:
# Optional post-core external transfer check and rank ablation.
RUN_ENHANCEMENTS = False
if RUN_ENHANCEMENTS:
    subprocess.run([sys.executable, 'scripts/run_external_eval.py', '--adapter', f'outputs/{RUN_ID}/training/adapter', '--output-dir', f'outputs/{RUN_ID}'], check=True)
    subprocess.run([sys.executable, 'scripts/run_ablation.py', '--data-dir', f'outputs/{RUN_ID}/data', '--output-dir', 'outputs/ablation_rank'], check=True)

KeyboardInterrupt: 

In [10]:
# Archive outputs before the hosted runtime is reclaimed.
archive = subprocess.check_output([sys.executable, '-c', "import shutil; print(shutil.make_archive('medical_qlora_outputs', 'zip', 'outputs'))"], text=True).strip()
print('Download this file:', archive)

Download this file: /content/medical-llm-qlora/medical_qlora_outputs.zip
